# 第9周-Day2-CNN基础与经典视觉骨干网络

🤖🧠 **今日学习目标：掌握CNN的核心组件原理，了解经典视觉模型的设计思想**

## 🎯 学习目标
- 理解CNN的四大核心层：卷积层、激活层、池化层、全连接层
- 掌握卷积操作的三个关键参数：滤波器大小、步长、填充
- 了解经典CNN模型的演进脉络：LeNet → AlexNet → VGGNet → ResNet
- 理解ResNet残差连接的突破性思想

## 🔄 昨日复习
- 计算机视觉四大核心任务：**分类、检测、分割、追踪**
- 复杂度递进：分类 < 检测 < 分割 < 追踪
- CNN是视觉任务的基础架构，几乎所有视觉模型都基于CNN或其变体

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()

plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print("使用字体:", font_name)

## 🔧 CNN核心组件

CNN（卷积神经网络）就像一个**特征提取流水线**，图像从输入到输出，每一层都在做不同层次的"加工"：

| 层 | 作用 | 类比 |
|---|------|------|
| 📥 输入层 | 接收原始图像 | 原材料入库 |
| 🔄 卷积层 | 用滤波器滑动提取特征 | 用放大镜检查细节 |
| ⚡ 激活层 | 引入非线性，让网络能学复杂模式 | 决定"这个特征够不够明显" |
| 📉 池化层 | 降维压缩，保留关键特征 | 压缩摘要 |
| 🔗 全连接层 | 整合所有特征，输出最终结果 | 综合判断做出决策 |

让我们逐一理解每一层的工作原理！

In [ ]:
# CNN流水线可视化
fig, ax = plt.subplots(1, 1, figsize=(16, 4))
ax.set_xlim(0, 16)
ax.set_ylim(0, 4)
ax.axis('off')
ax.set_title('CNN 特征提取流水线', fontsize=16, fontweight='bold', pad=20)

layers = [
    ('📥\n输入层\n224×224×3', '#E3F2FD'),
    ('🔄\n卷积层\n提取边缘/纹理', '#BBDEFB'),
    ('⚡\nReLU\n非线性激活', '#90CAF9'),
    ('📉\n池化层\n降维压缩', '#64B5F6'),
    ('🔄\n卷积层\n提取部件/形状', '#42A5F5'),
    ('⚡\nReLU\n非线性激活', '#2196F3'),
    ('📉\n池化层\n进一步压缩', '#1E88E5'),
    ('🔗\n全连接层\n整合判断', '#1565C0'),
    ('📊\n输出层\n分类结果', '#0D47A1'),
]

for i, (text, color) in enumerate(layers):
    x = i * 1.65 + 0.5
    rect = patches.FancyBboxPatch((x, 0.5), 1.4, 2.8, 
                                    boxstyle="round,pad=0.1", 
                                    facecolor=color, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + 0.7, 1.9, text, ha='center', va='center', fontsize=9, fontweight='bold')
    if i < len(layers) - 1:
        ax.annotate('', xy=(x + 1.5, 1.9), xytext=(x + 1.9, 1.9),
                    arrowprops=dict(arrowstyle='->', color='#333', lw=2))

plt.tight_layout()
plt.show()

## 🔄 卷积操作详解

卷积是CNN的核心操作。想象你拿着一个小窗口（滤波器/卷积核）在图片上滑动，每到一个位置就计算窗口内像素与卷积核的乘积之和。

### 三个关键参数

1. **滤波器大小（Kernel Size）**：窗口多大？常见3×3、5×5、7×7
   - 越大 → 感受野越大 → 捕获更大范围的特征
   - 越小 → 参数越少 → 更容易训练

2. **步长（Stride）**：每次滑动几格？
   - 步长=1：精细扫描，输出尺寸大
   - 步长=2：快速扫描，输出尺寸缩小一半

3. **填充（Padding）**：边缘怎么处理？
   - Valid（不填充）：输出尺寸会缩小
   - Same（补零填充）：输出尺寸与输入相同

In [ ]:
# 卷积操作模拟演示
def conv2d_manual(input_matrix, kernel, stride=1, padding=0):
    """手动实现2D卷积"""
    # 添加padding
    if padding > 0:
        padded = np.pad(input_matrix, padding, mode='constant')
    else:
        padded = input_matrix.copy()
    
    h_in, w_in = padded.shape
    h_k, w_k = kernel.shape
    h_out = (h_in - h_k) // stride + 1
    w_out = (w_in - w_k) // stride + 1
    
    output = np.zeros((h_out, w_out))
    for i in range(h_out):
        for j in range(w_out):
            region = padded[i*stride:i*stride+h_k, j*stride:j*stride+w_k]
            output[i, j] = np.sum(region * kernel)
    return output

# 创建一个简单的6×6输入图像
input_img = np.array([
    [10, 10, 10,  0,  0,  0],
    [10, 10, 10,  0,  0,  0],
    [10, 10, 10,  0,  0,  0],
    [ 0,  0,  0, 10, 10, 10],
    [ 0,  0,  0, 10, 10, 10],
    [ 0,  0,  0, 10, 10, 10],
])

# 边缘检测卷积核（检测竖直边缘）
edge_kernel = np.array([
    [-1,  0,  1],
    [-1,  0,  1],
    [-1,  0,  1],
])

# 执行卷积
output = conv2d_manual(input_img, edge_kernel)

# 可视化
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

ax1.imshow(input_img, cmap='gray', vmin=0, vmax=10)
ax1.set_title('输入图像 (6×6)', fontsize=13)
for i in range(6):
    for j in range(6):
        ax1.text(j, i, str(int(input_img[i, j])), ha='center', va='center', fontsize=10, 
                color='white' if input_img[i,j] > 5 else 'black')

ax2.imshow(edge_kernel, cmap='RdBu', vmin=-1, vmax=1)
ax2.set_title('卷积核 (3×3)\n竖直边缘检测', fontsize=13)
for i in range(3):
    for j in range(3):
        ax2.text(j, i, str(int(edge_kernel[i, j])), ha='center', va='center', fontsize=12, fontweight='bold')

im = ax3.imshow(output, cmap='RdBu')
ax3.set_title(f'卷积输出 (4×4)\n白=正边缘，蓝=负边缘', fontsize=13)
for i in range(output.shape[0]):
    for j in range(output.shape[1]):
        ax3.text(j, i, str(int(output[i, j])), ha='center', va='center', fontsize=10)

plt.suptitle('卷积操作演示：用3×3卷积核检测竖直边缘', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 观察：卷积核成功检测到了两个方块之间的竖直边缘！")

## ⚡ 激活函数与📉 池化层

### ReLU激活函数

ReLU（Rectified Linear Unit）是CNN中最常用的激活函数：**f(x) = max(0, x)**

- 正数不变，负数变0
- 为什么要非线性？如果全是线性操作，再多层网络也等价于一层线性变换
- ReLU好处：计算简单、梯度传播好、缓解梯度消失

### 池化层（Pooling）

池化层的作用是**降维压缩**，减少参数量和计算量：

- **最大池化**：取窗口内最大值 → 保留最显著特征
- **平均池化**：取窗口内平均值 → 保留整体信息
- **全局平均池化**：对整个特征图取平均 → 替代全连接层，减少参数

In [ ]:
# 激活函数对比
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ReLU
x = np.linspace(-5, 5, 100)
y_relu = np.maximum(0, x)
axes[0].plot(x, y_relu, 'b-', linewidth=3)
axes[0].axhline(y=0, color='gray', linewidth=0.5)
axes[0].axvline(x=0, color='gray', linewidth=0.5)
axes[0].set_title('ReLU: f(x) = max(0, x)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].fill_between(x, y_relu, alpha=0.1, color='blue')
axes[0].text(-3, 4, 'x<0 → 输出0\n（神经元"死亡"）', fontsize=11, color='red')
axes[0].text(2, 3, 'x>0 → 输出x\n（直接传递）', fontsize=11, color='blue')

# 池化操作演示
feature_map = np.array([
    [1, 3, 2, 1],
    [4, 6, 5, 2],
    [7, 8, 1, 0],
    [3, 2, 9, 4],
])

# 最大池化 2×2
def max_pool_2x2(fm):
    h, w = fm.shape
    out = np.zeros((h//2, w//2))
    for i in range(0, h, 2):
        for j in range(0, w, 2):
            out[i//2, j//2] = np.max(fm[i:i+2, j:j+2])
    return out

max_pooled = max_pool_2x2(feature_map)

axes[1].imshow(feature_map, cmap='YlOrRd', vmin=0, vmax=10)
axes[1].set_title('特征图 (4×4)', fontsize=14, fontweight='bold')
for i in range(4):
    for j in range(4):
        axes[1].text(j, i, str(int(feature_map[i, j])), ha='center', va='center', fontsize=14, fontweight='bold')
    if i < 3:
        axes[1].axhline(y=i+0.5, color='white', linewidth=2)
for j in range(3):
    axes[1].axvline(x=j+0.5, color='white', linewidth=2)

axes[2].imshow(max_pooled, cmap='YlOrRd', vmin=0, vmax=10)
axes[2].set_title('最大池化结果 (2×2)\n每块取最大值', fontsize=14, fontweight='bold')
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, str(int(max_pooled[i, j])), ha='center', va='center', fontsize=16, fontweight='bold')

plt.suptitle('激活函数与池化操作', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("💡 最大池化：4×4 → 2×2，参数量减少75%，但保留了每个区域最显著的特征")

## 🏛️ 经典CNN模型演进

CNN的发展史就是一部"如何让网络越来越深"的故事：

```
LeNet (1998) → AlexNet (2012) → VGGNet (2014) → ResNet (2015)
  5层            8层              19层            152层+
```

In [ ]:
# 经典CNN模型演进可视化
fig, ax = plt.subplots(1, 1, figsize=(16, 8))
ax.axis('off')
ax.set_title('经典CNN模型演进时间线', fontsize=18, fontweight='bold', pad=20)

# 时间线
timeline_y = 0.5
ax.plot([0.5, 14], [timeline_y, timeline_y], 'k-', linewidth=3)

models = [
    (1.5, 1998, 'LeNet-5', '5层\n手写数字识别\n开创CNN先河', '#4CAF50', 0.7),
    (4.5, 2012, 'AlexNet', '8层\nImageNet冠军\nGPU训练时代开启', '#2196F3', 0.85),
    (7.5, 2014, 'VGGNet', '19层\n统一3×3卷积\n更深更简洁', '#FF9800', 0.92),
    (10.5, 2015, 'ResNet', '152层\n残差连接\n解决退化问题', '#E91E63', 1.0),
    (13.0, 2020, 'EfficientNet', '复合缩放\n精度+效率平衡\n移动端友好', '#9C27B0', 1.05),
]

for x, year, name, desc, color, height in models:
    # 年份标记
    ax.plot(x, timeline_y, 'o', color=color, markersize=15, zorder=5)
    ax.text(x, timeline_y - 0.08, str(year), ha='center', fontsize=11, fontweight='bold')
    # 模型信息卡片
    rect = patches.FancyBboxPatch((x-1.1, timeline_y + 0.1), 2.2, height,
                                    boxstyle="round,pad=0.15",
                                    facecolor=color, alpha=0.15,
                                    edgecolor=color, linewidth=2)
    ax.add_patch(rect)
    ax.text(x, timeline_y + 0.1 + height/2 + 0.05, 
            f'**{name}**\n{desc}', 
            ha='center', va='center', fontsize=10)

plt.tight_layout()
plt.show()

### 📖 四大经典模型要点

#### 1. LeNet-5 (1998) — CNN的开山之作
- **作者**：Yann LeCun
- **贡献**：第一个成功应用的CNN
- **应用**：银行支票手写数字识别、邮政编码识别
- **结构**：2个卷积层 + 3个全连接层，仅约6万参数
- **局限性**：只能处理小图片（28×28），网络很浅

#### 2. AlexNet (2012) — 深度学习的引爆点
- **贡献**：在ImageNet大赛上将错误率从26%降到15%，碾压第二名
- **突破**：
  - 首次使用 **GPU** 并行训练（两块GTX 580）
  - 引入 **ReLU** 激活函数（取代Sigmoid）
  - 使用 **Dropout** 防止过拟合
  - **数据增强**（随机裁剪、水平翻转）
- **结构**：5个卷积层 + 3个全连接层，约6000万参数

#### 3. VGGNet (2014) — 更深更好
- **核心思想**：全部使用 **3×3 小卷积核**，通过堆叠增加深度
- **关键洞察**：两个3×3卷积 = 一个5×5卷积的感受野，但参数更少！
  - 3×3×3 = 27 参数 vs 5×5 = 25 参数
  - 但两个3×3多了一次ReLU，表达能力更强
- **结构**：VGG-16有16层，约1.38亿参数
- **影响**：3×3卷积成为后续几乎所有模型的"标配"

#### 4. ResNet (2015) — 残差学习的革命
- **核心问题**：网络越深，训练越困难（梯度消失/退化）
- **核心思想**：**跳跃连接（Skip Connection）**
  - 如果某一层没有帮助，可以直接"跳过"它
  - 学习的是 F(x) = H(x) - x（残差），而不是直接学 H(x)
- **影响**：让训练152层甚至1000层网络成为可能
- **意义**：彻底解决了深度网络的退化问题

In [ ]:
# ResNet残差连接原理可视化
fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.axis('off')
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.set_title('ResNet 残差块（Residual Block）原理', fontsize=16, fontweight='bold', pad=20)

# 残差块结构
# 输入x
ax.text(1, 7, '输入 x', fontsize=14, fontweight='bold', ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#E3F2FD', edgecolor='#1565C0'))

# 主路径
ax.annotate('', xy=(3, 7), xytext=(2, 7), arrowprops=dict(arrowstyle='->', lw=2.5, color='#1565C0'))

# 卷积层1
conv1 = patches.FancyBboxPatch((3, 6), 2.5, 2, boxstyle="round,pad=0.2",
                                facecolor='#BBDEFB', edgecolor='#1565C0', linewidth=2)
ax.add_patch(conv1)
ax.text(4.25, 7, 'Conv + BN + ReLU\n权重层 F₁(x)', ha='center', va='center', fontsize=11, fontweight='bold')

ax.annotate('', xy=(6, 7), xytext=(5.5, 7), arrowprops=dict(arrowstyle='->', lw=2.5, color='#1565C0'))

# 卷积层2
conv2 = patches.FancyBboxPatch((6, 6), 2.5, 2, boxstyle="round,pad=0.2",
                                facecolor='#90CAF9', edgecolor='#1565C0', linewidth=2)
ax.add_patch(conv2)
ax.text(7.25, 7, 'Conv + BN\n权重层 F₂(x)', ha='center', va='center', fontsize=11, fontweight='bold')

# 主路径到加法
ax.annotate('', xy=(9.5, 7), xytext=(8.5, 7), arrowprops=dict(arrowstyle='->', lw=2.5, color='#1565C0'))

# 跳跃连接
ax.annotate('', xy=(9.5, 7), xytext=(2, 7), 
            arrowprops=dict(arrowstyle='->', lw=2.5, color='#E91E63', 
                           connectionstyle='arc3,rad=0.4'))
ax.text(5.5, 8.8, '✨ 跳跃连接（Shortcut/Skip Connection）', 
        fontsize=12, color='#E91E63', fontweight='bold', ha='center',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#FCE4EC', edgecolor='#E91E63'))

# 加法
add = patches.FancyBboxPatch((9.5, 6.2), 1.5, 1.6, boxstyle="round,pad=0.2",
                               facecolor='#FFF9C4', edgecolor='#F57F17', linewidth=2)
ax.add_patch(add)
ax.text(10.25, 7, '➕\nF(x)+x', ha='center', va='center', fontsize=12, fontweight='bold')

# 输出
ax.annotate('', xy=(12.5, 7), xytext=(11, 7), arrowprops=dict(arrowstyle='->', lw=2.5, color='#1565C0'))
ax.text(13, 7, '输出\nReLU(F(x)+x)', fontsize=13, fontweight='bold', ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#C8E6C9', edgecolor='#2E7D32'))

# 公式说明
ax.text(7, 4, '数学表达：', fontsize=14, fontweight='bold', ha='center')
ax.text(7, 3, '普通网络：H(x) = F(x)       直接学习映射', fontsize=13, ha='center', color='#666')
ax.text(7, 2, '残差网络：F(x) = H(x) - x    学习残差（差值）', fontsize=13, ha='center', color='#1565C0', fontweight='bold')
ax.text(7, 1, '💡 如果这层不需要，F(x)→0，等价于恒等映射，不会比浅层网络差！', 
        fontsize=12, ha='center', color='#E91E63',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF3E0', edgecolor='#E91E63'))

plt.tight_layout()
plt.show()

## 📊 四大模型参数对比

In [ ]:
# 模型对比表
models_data = {
    '模型': ['LeNet-5', 'AlexNet', 'VGG-16', 'ResNet-50', 'ResNet-152'],
    '年份': [1998, 2012, 2014, 2015, 2015],
    '层数': [5, 8, 16, 50, 152],
    '参数量(M)': [0.06, 60, 138, 25.6, 60.2],
    'Top-5错误率(%)': [None, 16.4, 7.3, 3.6, 3.6],
    '突破点': ['CNN开山之作', 'GPU+ReLU+Dropout', '统一3×3卷积', '残差连接解决退化', '更深的残差网络'],
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# 左图：层数和参数量对比
x = np.arange(len(models_data['模型']))
width = 0.35

bars1 = ax1.bar(x - width/2, models_data['层数'], width, label='层数', color='#42A5F5', alpha=0.8)
ax1_twin = ax1.twinx()
bars2 = ax1_twin.bar(x + width/2, models_data['参数量(M)'], width, label='参数量(M)', color='#EF5350', alpha=0.8)

ax1.set_xlabel('模型')
ax1.set_ylabel('层数', color='#42A5F5')
ax1_twin.set_ylabel('参数量 (百万)', color='#EF5350')
ax1.set_xticks(x)
ax1.set_xticklabels(models_data['模型'], fontsize=11)
ax1.set_title('CNN模型复杂度演进', fontsize=14, fontweight='bold')

for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1, 
             str(int(bar.get_height())), ha='center', va='bottom', fontsize=10, fontweight='bold')

# 右图：错误率趋势（排除LeNet）
years_err = [2012, 2014, 2015, 2015]
errors = [16.4, 7.3, 3.6, 3.6]
names_err = ['AlexNet', 'VGG-16', 'ResNet-50', 'ResNet-152']

colors = ['#42A5F5', '#FF9800', '#E91E63', '#9C27B0']
ax2.plot(years_err, errors, 'o-', color='#333', linewidth=2, markersize=8, zorder=3)
for i, (y, e, n) in enumerate(zip(years_err, errors, names_err)):
    ax2.annotate(f'{n}\n{e}%', xy=(y, e), xytext=(y+0.1, e+2), 
                fontsize=11, fontweight='bold', color=colors[i],
                arrowprops=dict(arrowstyle='->', color=colors[i]))

ax2.set_xlabel('年份')
ax2.set_ylabel('Top-5 错误率 (%)')
ax2.set_title('ImageNet 分类错误率变化', fontsize=14, fontweight='bold')
ax2.set_ylim(0, 20)
ax2.grid(True, alpha=0.3)

plt.suptitle('经典CNN模型对比', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n📊 关键发现：")
print("  • ResNet-50 用更少的参数（25.6M vs VGG的138M）达到了更低的错误率")
print("  • 残差连接让网络从19层直接跳到152层，错误率却更低")
print("  • 深度不是目的，关键是要能训练得动")

## 🧠 从生物学视角看CNN

你有没有想过，CNN的设计灵感其实来自大脑的**视觉皮层**？

### 🧬 生物视觉 vs CNN

| 生物视觉皮层 | CNN对应 | 共同点 |
|-------------|---------|--------|
| 视网膜感受野 | 卷积核（滤波器） | 局部区域感知 |
| 简单细胞 → 复杂细胞 | 浅层 → 深层特征 | 层次化特征提取 |
| V1区检测边缘/方向 | 第1层卷积提取边缘 | 底层检测简单特征 |
| V2/V4区识别形状/纹理 | 中间层检测部件 | 中层组合简单特征 |
| IT区识别物体 | 全连接层做分类 | 高层做整体判断 |

**Hubel & Wiesel（1959）**发现猫的视觉皮层神经元对特定方向的边缘有选择性响应，这正是卷积核的生物学原型！他们因此获得了1981年诺贝尔生理学或医学奖。

In [ ]:
# 特征层次可视化：浅层→深层
fig, ax = plt.subplots(1, 1, figsize=(15, 6))
ax.axis('off')
ax.set_title('CNN特征提取的层次结构（类比视觉皮层）', fontsize=16, fontweight='bold', pad=20)

# 用颜色方块模拟不同层次的特征
layers_info = [
    ('输入图像', '原始像素', '#E0E0E0', None),
    ('第1层\n(浅层卷积)', '边缘、颜色、纹理\n≈ V1区简单细胞', '#90CAF9'),
    ('第2-3层\n(中层卷积)', '眼睛、鼻子、角\n≈ V2/V4区复杂细胞', '#64B5F6'),
    ('第4-5层\n(深层卷积)', '人脸、车轮、建筑\n≈ IT区物体识别', '#2196F3'),
    ('全连接层\n(分类)', '最终分类结果', '#0D47A1'),
]

for i, (name, desc, color, _) in enumerate(layers_info):
    x = i * 2.8 + 0.5
    # 渐进缩小的方块（模拟特征图尺寸变小，通道数变多）
    h = 4.5 - i * 0.4
    w = 1.8 - i * 0.1
    y_center = 3
    rect = patches.FancyBboxPatch((x, y_center - h/2), w, h,
                                    boxstyle="round,pad=0.15",
                                    facecolor=color, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y_center + 0.3, name, ha='center', va='center', fontsize=10, fontweight='bold', color='white' if i >= 3 else 'black')
    ax.text(x + w/2, y_center - 0.5, desc, ha='center', va='center', fontsize=9, color='white' if i >= 3 else '#333')
    
    if i < len(layers_info) - 1:
        next_x = (i + 1) * 2.8 + 0.5
        ax.annotate('', xy=(next_x, y_center), xytext=(x + w + 0.1, y_center),
                    arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# 感受野箭头
ax.annotate('感受野\n越来越大', xy=(13, 5.2), fontsize=12, color='#E91E63', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#E91E63', lw=2), xytext=(2, 5.5))

ax.annotate('抽象程度\n越来越高', xy=(13, 0.8), fontsize=12, color='#4CAF50', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#4CAF50', lw=2), xytext=(2, 0.5))

plt.tight_layout()
plt.show()

## 🧪 实践练习

### 练习1：计算卷积输出尺寸

卷积输出尺寸公式：**Output = ⌊(Input - Kernel + 2×Padding) / Stride⌋ + 1**

In [ ]:
# 卷积输出尺寸计算器
def calc_conv_output(input_size, kernel_size, padding=0, stride=1):
    output = (input_size - kernel_size + 2 * padding) // stride + 1
    return output

print("=" * 50)
print('📐 卷积输出尺寸计算练习')
print("=" * 50)

# 常见配置
configs = [
    (32, 3, 0, 1, '无填充，步长1'),
    (32, 3, 1, 1, 'Same填充，步长1'),
    (32, 5, 0, 2, '无填充，步长2'),
    (224, 7, 3, 2, 'AlexNet第一层配置'),
    (112, 3, 1, 1, 'VGG网络中间层'),
]

for inp, k, p, s, desc in configs:
    out = calc_conv_output(inp, k, p, s)
    print(f'\n  {desc}:')
    print(f'    输入={inp}×{inp}, 卷积核={k}×{k}, 填充={p}, 步长={s}')
    print(f'    输出 = ({inp} - {k} + 2×{p}) / {s} + 1 = {out}×{out}')

print(f"\n{'=' * 50}")
print("✏️ 自己试试：输入=64×64, 卷积核=5×5, padding=2, stride=1 → 输出=?")
print(f"   答案：{calc_conv_output(64, 5, 2, 1)}×{calc_conv_output(64, 5, 2, 1)}")

### 练习2：模拟VGGNet的卷积过程

让我们追踪一张224×224图片经过VGGNet前几层的尺寸变化：

In [ ]:
# VGGNet 特征图尺寸追踪
print("=" * 55)
print("🧱 VGG-16 各层特征图尺寸追踪")
print("=" * 55)

layers = [
    ('输入',        224, 224,   3,  '-'),
    ('Conv3-64',    224, 224,  64,  '3×3, pad=1, stride=1'),
    ('Conv3-64',    224, 224,  64,  '3×3, pad=1, stride=1'),
    ('MaxPool',     112, 112,  64,  '2×2, stride=2'),
    ('Conv3-128',   112, 112, 128,  '3×3, pad=1, stride=1'),
    ('Conv3-128',   112, 112, 128,  '3×3, pad=1, stride=1'),
    ('MaxPool',      56,  56, 128,  '2×2, stride=2'),
    ('Conv3-256',    56,  56, 256,  '3×3, pad=1, stride=1'),
    ('Conv3-256',    56,  56, 256,  '3×3, pad=1, stride=1'),
    ('Conv3-256',    56,  56, 256,  '3×3, pad=1, stride=1'),
    ('MaxPool',      28,  28, 256,  '2×2, stride=2'),
    ('Conv3-512',    28,  28, 512,  '3×3, pad=1, stride=1'),
    ('Conv3-512',    28,  28, 512,  '3×3, pad=1, stride=1'),
    ('Conv3-512',    28,  28, 512,  '3×3, pad=1, stride=1'),
    ('MaxPool',      14,  14, 512,  '2×2, stride=2'),
    ('Conv3-512',    14,  14, 512,  '3×3, pad=1, stride=1'),
    ('Conv3-512',    14,  14, 512,  '3×3, pad=1, stride=1'),
    ('Conv3-512',    14,  14, 512,  '3×3, pad=1, stride=1'),
    ('MaxPool',       7,   7, 512,  '2×2, stride=2'),
    ('FC-4096',       7,   7, 4096, '全连接层'),
    ('FC-4096',       7,   7, 4096, '全连接层'),
    ('FC-1000',       7,   7, 1000, '1000类输出'),
]

print(f'{"层":<12} {"尺寸":<12} {"通道数":<8} {"配置":<25}')
print('-' * 55)
for name, h, w, ch, cfg in layers:
    print(f'{name:<12} {h}×{w}×{ch:<4} {cfg}')

print("\n💡 观察规律：")
print('  • 卷积层(pad=1)：尺寸不变，通道数增加')
print('  • 池化层(2×2)：尺寸减半，通道数不变')
print('  • 空间越来越小，但特征越来越"厚"（通道越来越多）')

## 📝 课后测试

### ❶ 卷积核的作用（单选）
一个3×3的边缘检测卷积核 `[[-1,0,1],[-1,0,1],[-1,0,1]]` 最擅长检测什么方向的特征？
- A) 水平边缘
- B) **竖直边缘**
- C) 对角线
- D) 角点

### ❷ 池化层的作用（单选）
以下哪项不是池化层的作用？
- A) 减少参数量
- B) **引入非线性**
- C) 增加感受野
- D) 提供平移不变性

### ❸ ResNet的核心创新（简答）
ResNet解决了什么问题？它是如何解决的？

### ❹ 计算题
输入图像为 64×64×3，使用 32个 5×5 卷积核，padding=2, stride=1。输出尺寸是多少？参数量是多少？

### ❺ 思考题
为什么VGGNet选择用两个3×3卷积代替一个5×5卷积？这样做的优势是什么？

**回复答案我帮你批改 ✅**

## 🎓 学习总结

### 今日核心要点

1. **CNN四大组件**：卷积（提取特征）→ ReLU（引入非线性）→ 池化（降维）→ 全连接（分类）
2. **卷积三参数**：滤波器大小、步长、填充，决定输出尺寸和感受野
3. **模型演进核心**：从5层到152层，关键突破是**残差连接**
4. **ResNet精髓**：学习残差 F(x)=H(x)-x，让深层网络也能轻松训练
5. **生物学启发**：CNN的层次特征提取 ≈ 视觉皮层的简单细胞→复杂细胞→IT区

### 知识脉络

```
边缘/纹理 → 眼睛/鼻子 → 人脸/物体 → 分类结果
  (浅层)      (中层)      (深层)     (全连接)
```

In [ ]:
# 本周学习进度可视化
fig, ax = plt.subplots(1, 1, figsize=(14, 5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 3)
ax.axis('off')
ax.set_title('第9周学习进度', fontsize=16, fontweight='bold', pad=15)

days = [
    (1, 'Day1', '计算机视觉全景', '#4CAF50', True),
    (3, 'Day2', 'CNN基础与经典视觉骨干网络', '#4CAF50', True),
    (5, 'Day3', '目标检测：YOLO与Faster R-CNN', '#BDBDBD', False),
    (7, 'Day4', '图像分割：语义分割与实例分割', '#BDBDBD', False),
    (9, 'Day5', '视觉Transformer：ViT与Swin', '#BDBDBD', False),
]

for x, label, topic, color, done in days:
    circle = plt.Circle((x, 1.5), 0.6, color=color, alpha=0.3 if not done else 0.8)
    ax.add_patch(circle)
    ax.text(x, 1.7, label, ha='center', va='center', fontsize=12, fontweight='bold')
    ax.text(x, 1.2, topic, ha='center', va='center', fontsize=9, 
            color='#333' if done else '#999')
    if done:
        ax.text(x, 2.3, '✅', ha='center', fontsize=16)

# 连接线
for i in range(len(days) - 1):
    x1 = days[i][0] + 0.6
    x2 = days[i+1][0] - 0.6
    ax.plot([x1, x2], [1.5, 1.5], '--', color='#BDBDBD' if not days[i+1][4] else '#4CAF50', linewidth=2)

plt.tight_layout()
plt.show()

print("\n📊 进度：第9周/12 | Day2/7 | 当前：计算机视觉")

## 🔑 今日英文术语

| 术语 | 音标 | 中文释义 |
|------|------|----------|
| **Convolution** | /ˌkɒnvəˈluːʃən/ | 卷积 |
| **Kernel / Filter** | /ˈkɜːnl/ | 卷积核/滤波器 |
| **Stride** | /straɪd/ | 步长 |
| **Padding** | /ˈpædɪŋ/ | 填充 |
| **Pooling** | /ˈpuːlɪŋ/ | 池化 |
| **Residual Connection** | /rɪˈzɪdjuəl/ | 残差连接/跳跃连接 |
| **Skip Connection** | /skɪp/ | 跳跃连接（同Residual） |
| **Backbone** | /ˈbækbəʊn/ | 骨干网络 |
| **Feature Map** | /ˈfiːtʃə mæp/ | 特征图 |
| **Receptive Field** | /rɪˈseptɪv fiːld/ | 感受野 |

## 🔄 往期回顾

昨天我们学了计算机视觉四大任务，今天来回顾一道题：

**问**：图像分割和目标检测最大的区别是什么？

**答**：目标检测用矩形框框出目标位置和类别（"哪里有什么"），图像分割则做到**像素级别**的分类（每个像素属于什么类别）。分割比检测更精细，但计算量也更大。

💡 下一节课我们会学习**目标检测**（YOLO、Faster R-CNN），它是连接分类和分割的桥梁！